# LCD Analysis Dataset Cleaning and Validation

This notebook cleans and validates the LCD-based daily weather dataset for the analysis period (2015–2025).

## Importing Necessities

In [1]:
import pandas as pd
from pathlib import Path

## File paths

In [4]:
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

input_file = DATA_PROCESSED / 'lcd_analysis_2015_2025_daily.csv'
output_file = DATA_PROCESSED / 'lcd_analysis_2015_2025_daily_clean.csv'

## Loading the dataset

In [5]:
df = pd.read_csv(input_file)
print('Shape:', df.shape)
df.head()


Shape: (240739, 12)


,parish,lcd_station_id,lcd_station_name,station_latitude,station_longitude,date,avg_temp,avg_dew_point,avg_relative_humidity,avg_wind_speed,max_temp,min_temp
0,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN
1,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-03,NaN,NaN,NaN,NaN,NaN,NaN
3,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-04,NaN,NaN,NaN,NaN,NaN,NaN
4,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN


## Inspecting column names and data types

In [6]:
print(df.columns.tolist())
print('\nData types before cleaning:')
print(df.dtypes)


['parish', 'lcd_station_id', 'lcd_station_name', 'station_latitude', 'station_longitude', 'date', 'avg_temp', 'avg_dew_point', 'avg_relative_humidity', 'avg_wind_speed', 'max_temp', 'min_temp']

Data types before cleaning:
parish                    object
lcd_station_id            object
lcd_station_name          object
station_latitude         float64
station_longitude        float64
date                      object
avg_temp                  object
avg_dew_point            float64
avg_relative_humidity    float64
avg_wind_speed           float64
max_temp                  object
min_temp                  object
dtype: object


## Converting key columns to proper types

In [7]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

numeric_cols = [
    'station_latitude',
    'station_longitude',
    'avg_temp',
    'avg_dew_point',
    'avg_relative_humidity',
    'avg_wind_speed',
    'max_temp',
    'min_temp'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.dtypes)


parish                           object
lcd_station_id                   object
lcd_station_name                 object
station_latitude                float64
station_longitude               float64
date                     datetime64[ns]
avg_temp                        float64
avg_dew_point                   float64
avg_relative_humidity           float64
avg_wind_speed                  float64
max_temp                        float64
min_temp                        float64
dtype: object


## Missingness summary after type conversion

In [8]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
}).sort_values('missing_percent', ascending=False)

missing_summary


,missing_count,missing_percent
avg_dew_point,158140,65.689398
avg_relative_humidity,157805,65.550243
avg_wind_speed,141102,58.612024
avg_temp,132311,54.960351
min_temp,132248,54.934182
max_temp,132218,54.921720
lcd_station_id,0,0.000000
parish,0,0.000000
station_latitude,0,0.000000
lcd_station_name,0,0.000000


## Missingness summary for the key weather variables only

In [13]:
weather_cols = [
    'avg_temp',
    'avg_dew_point',
    'avg_relative_humidity',
    'avg_wind_speed',
    'max_temp',
    'min_temp'
]

weather_missing = pd.DataFrame({
    'missing_count': df[weather_cols].isna().sum(),
    'missing_percent': df[weather_cols].isna().mean() * 100
}).sort_values('missing_percent', ascending=False)

weather_missing

,missing_count,missing_percent
avg_dew_point,158140,65.689398
avg_relative_humidity,157805,65.550243
avg_wind_speed,141102,58.612024
avg_temp,132311,54.960351
min_temp,132248,54.934182
max_temp,132218,54.921720


## Checking for duplicate parish-date rows

In [10]:
dup_count = df.duplicated(subset=['parish', 'date']).sum()
print('Duplicate parish-date rows:', dup_count)

Duplicate parish-date rows: 0


## Inspecting a few rows with missing key weather fields

In [11]:
df[df[weather_cols].isna().any(axis=1)].head(20)


,parish,lcd_station_id,lcd_station_name,station_latitude,station_longitude,date,avg_temp,avg_dew_point,avg_relative_humidity,avg_wind_speed,max_temp,min_temp
0,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN
1,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-03,NaN,NaN,NaN,NaN,NaN,NaN
3,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-04,NaN,NaN,NaN,NaN,NaN,NaN
4,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN
5,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-06,NaN,NaN,NaN,NaN,NaN,NaN
6,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN
7,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-08,NaN,NaN,NaN,NaN,NaN,NaN
8,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-09,NaN,NaN,NaN,NaN,NaN,NaN
9,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-10,NaN,NaN,NaN,NaN,NaN,NaN


## Basic range checks for key variables

In [12]:
range_summary = pd.DataFrame({
    'min': df[weather_cols].min(),
    'max': df[weather_cols].max(),
    'mean': df[weather_cols].mean()
})

range_summary

,min,max,mean
avg_temp,-9.9,35.9,20.481900
avg_dew_point,-16.7,29.4,14.973828
avg_relative_humidity,24.0,100.0,73.652953
avg_wind_speed,0.0,19.2,2.659740
max_temp,-7.1,43.3,26.208384
min_temp,-17.1,31.1,14.702668
